# Demo Quantitative Stock Selection Model: Analyst Forecast Dispersion

Vamos 

The difficulty in generating this signal resides in merging data from two sources, the analyst forecasts and the prices. We will use **merge** function from pandas but, to do that, we need to generate a unique key for each row in both dataframes.

## Merge Two Data Frames by Keys

In [1]:
import pandas as pd
import numpy as np
df1 = pd.DataFrame({'key': ['A', 'B', 'C', 'D'],
                    'value1': [1, 2, 3, 5]})
df2 = pd.DataFrame({'key': ['E', 'D', 'C', 'B'],
                    'value2': [5, 6, 7, 8]})
display(df1)
display(df2)

,key,value1
0,A,1
1,B,2
2,C,3
3,D,5


,key,value2
0,E,5
1,D,6
2,C,7
3,B,8


Merge the two data frames on column "key". Use intersection of keys from both data frames. Preserve the order of keys from the left frame.

In [2]:
df3=pd.merge(df1,df2,on="key",how="inner")
df3

,key,value1,value2
0,B,2,8
1,C,3,7
2,D,5,6


Merge the two data frames on column "key". Use all keys from the left frame. Preserve the order of keys from the left frame.

In [3]:
#NaN will show up when the key is not available in the right frame
df3=pd.merge(df1,df2,on="key",how="left")
df3

,key,value1,value2
0,A,1,NaN
1,B,2,8.0
2,C,3,7.0
3,D,5,6.0


For more instruction about merge function, see https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.merge.html

## Analyst Forecast Dispersion 

$$
\frac{\text{STD of Forecast }{t}}{\text{Price at }t}
$$

## Step 1: Generate Keys to Merge Data Frames

In [4]:
df1 = pd.read_csv('classdata/IBES.csv')

In [5]:
#We only select the columns we need
df1=df1[["CUSIP","STATPERS","NUMEST","MEANEST","STDEV"]]
df1.head()

,CUSIP,STATPERS,NUMEST,MEANEST,STDEV
0,87482X10,20130218,10,2.42,0.09
1,87482X10,20130318,16,2.20,0.07
2,87482X10,20130418,19,1.15,0.04
3,87482X10,20130518,23,1.92,0.08
4,87482X10,20130618,4,2.41,0.02


Remove the records with no more than two analyst forecasts.

In [6]:
df1=df1[df1.NUMEST>2]

We generate the unique key for each row in df1 as "CUSIP"+"STATPERS", which represents the stock and the date from which this row is generated.

In [7]:
df1=df1[["CUSIP","STATPERS","STDEV"]]
df1["key"]=df1["CUSIP"]+df1["STATPERS"].astype(str)
df1.head()

,CUSIP,STATPERS,STDEV,key
0,87482X10,20130218,0.09,87482X1020130218
1,87482X10,20130318,0.07,87482X1020130318
2,87482X10,20130418,0.04,87482X1020130418
3,87482X10,20130518,0.08,87482X1020130518
4,87482X10,20130618,0.02,87482X1020130618


We then load the data file that contains the last closing price available to IBES before the statistics (STDEV) were calculated. 

In [8]:
df2 = pd.read_csv('classdata/ActualPrice.csv')
df2 = df2[["CUSIP","STATPERS","PRICE"]]
df2.head()

,CUSIP,STATPERS,PRICE
0,87482X10,20130218,148.37
1,87482X10,20130318,150.42
2,87482X10,20130418,152.11
3,87482X10,20130518,152.80
4,87482X10,20130618,156.46


Remove the records that have zero in prices.

In [9]:
df2=df2[df2.PRICE>0]

We generate the unique key for each row in df2 similarly.

In [10]:
df2["key"]=df2["CUSIP"]+df2["STATPERS"].astype(str)
df2=df2[["key","PRICE"]]
df2.head()

,key,PRICE
0,87482X1020130218,148.37
1,87482X1020130318,150.42
2,87482X1020130418,152.11
3,87482X1020130518,152.80
4,87482X1020130618,156.46


Merge the two data frames (df1 and df2) on column "key". Use intersection of keys from both data frames. Preserve the order of the left keys.

In [11]:
df3=pd.merge(df1,df2,on="key",how="inner")
df3.head()

,CUSIP,STATPERS,STDEV,key,PRICE
0,87482X10,20130218,0.09,87482X1020130218,148.37
1,87482X10,20130318,0.07,87482X1020130318,150.42
2,87482X10,20130418,0.04,87482X1020130418,152.11
3,87482X10,20130518,0.08,87482X1020130518,152.80
4,87482X10,20130618,0.02,87482X1020130618,156.46


## Step 2: Generate Signals and Available Quarters

Simply generate the signal in a new column.

In [12]:
df3["Signal"]=df3["STDEV"]/df3["PRICE"]
df3.head()

,CUSIP,STATPERS,STDEV,key,PRICE,Signal
0,87482X10,20130218,0.09,87482X1020130218,148.37,0.000607
1,87482X10,20130318,0.07,87482X1020130318,150.42,0.000465
2,87482X10,20130418,0.04,87482X1020130418,152.11,0.000263
3,87482X10,20130518,0.08,87482X1020130518,152.80,0.000524
4,87482X10,20130618,0.02,87482X1020130618,156.46,0.000128


In [13]:
#These libraries are loaded to move each date six months forwards (to the end of the returned month)
from dateutil.relativedelta import relativedelta

Generate **yyyymm** as $100\times year + 3\times quarter$ using list comprehension.

In [14]:
df3['STATPERS']=pd.to_datetime(df3['STATPERS'],format="%Y%m%d")
df3["year"]=[s.year for s in df3["STATPERS"]]
df3["quarter"]=[s.quarter for s in df3["STATPERS"]]
df3["Date"]=df3["year"]*100+df3["quarter"]*3
df3.head()

,CUSIP,STATPERS,STDEV,key,PRICE,Signal,year,quarter,Date
0,87482X10,2013-02-18,0.09,87482X1020130218,148.37,0.000607,2013,1,201303
1,87482X10,2013-03-18,0.07,87482X1020130318,150.42,0.000465,2013,1,201303
2,87482X10,2013-04-18,0.04,87482X1020130418,152.11,0.000263,2013,2,201306
3,87482X10,2013-05-18,0.08,87482X1020130518,152.80,0.000524,2013,2,201306
4,87482X10,2013-06-18,0.02,87482X1020130618,156.46,0.000128,2013,2,201306


Remove nan in df3 and sort df3.

In [15]:
df3.dropna(how='any',axis=0, inplace=True)
df3.sort_values(by=['CUSIP', 'Date'],inplace=True)
df3.reset_index(drop=True, inplace=True)

## Step 3: Generate Summary Statistics

For signals constructed using IBES data on analyst forecasts, the data are available at monthly frequency. We must only keep the signal from the last month of a quarter. For example, the standard deviation of analyst forecast is available in Jan, 
Feb, and Mar of 2012. To construct the signal of analyst forecast dispersion for 201203, you only need the standard deviation measured in March 2012.   

In [16]:
df3.groupby(["CUSIP"]).get_group("87482X10")

,CUSIP,STATPERS,STDEV,key,PRICE,Signal,year,quarter,Date
3478,87482X10,2013-02-18,0.09,87482X1020130218,148.37,0.000607,2013,1,201303
3479,87482X10,2013-03-18,0.07,87482X1020130318,150.42,0.000465,2013,1,201303
3480,87482X10,2013-04-18,0.04,87482X1020130418,152.11,0.000263,2013,2,201306
3481,87482X10,2013-05-18,0.08,87482X1020130518,152.80,0.000524,2013,2,201306
3482,87482X10,2013-06-18,0.02,87482X1020130618,156.46,0.000128,2013,2,201306
3483,87482X10,2013-07-18,0.05,87482X1020130718,150.10,0.000333,2013,3,201309
3484,87482X10,2013-08-18,0.02,87482X1020130818,153.55,0.000130,2013,3,201309
3485,87482X10,2013-09-18,0.02,87482X1020130918,157.52,0.000127,2013,3,201309
3486,87482X10,2013-10-18,0.02,87482X1020131018,155.75,0.000128,2013,4,201312
3487,87482X10,2013-11-18,0.06,87482X1020131118,156.75,0.000383,2013,4,201312


We use **groupby** to group the rows with the same "CUSIP" and "Date". Then we use **.tail(1)** to only keep the last row, which is the latest month in each quarter (the one we are supposed to use).

In [17]:
df3.groupby(["CUSIP","Date"]).get_group(("87482X10",201412))

,CUSIP,STATPERS,STDEV,key,PRICE,Signal,year,quarter,Date
3498,87482X10,2014-10-18,0.02,87482X1020141018,151.22,0.000132,2014,4,201412
3499,87482X10,2014-11-18,0.07,87482X1020141118,153.42,0.000456,2014,4,201412
3500,87482X10,2014-12-18,0.07,87482X1020141218,163.17,0.000429,2014,4,201412


In [18]:
df3.groupby(["CUSIP","Date"]).get_group(("87482X10",201412)).tail(1)

,CUSIP,STATPERS,STDEV,key,PRICE,Signal,year,quarter,Date
3500,87482X10,2014-12-18,0.07,87482X1020141218,163.17,0.000429,2014,4,201412


By removing **.get_group(("87482X10",201412))** in the code above, we can remove duplicated Dates in all groups.

In [19]:
df3=df3.groupby(["CUSIP","Date"]).tail(1)
df3.reset_index(drop=True, inplace=True)

Let's check if this works.

In [20]:
df3.groupby(["CUSIP"]).get_group("87482X10")

,CUSIP,STATPERS,STDEV,key,PRICE,Signal,year,quarter,Date
1184,87482X10,2013-03-18,0.07,87482X1020130318,150.42,0.000465,2013,1,201303
1185,87482X10,2013-06-18,0.02,87482X1020130618,156.46,0.000128,2013,2,201306
1186,87482X10,2013-09-18,0.02,87482X1020130918,157.52,0.000127,2013,3,201309
1187,87482X10,2013-12-18,0.09,87482X1020131218,155.01,0.000581,2013,4,201312
1188,87482X10,2014-03-18,0.03,87482X1020140318,147.24,0.000204,2014,1,201403
1189,87482X10,2014-06-18,0.05,87482X1020140618,140.78,0.000355,2014,2,201406
1190,87482X10,2014-09-18,0.08,87482X1020140918,152.18,0.000526,2014,3,201409
1191,87482X10,2014-12-18,0.07,87482X1020141218,163.17,0.000429,2014,4,201412
1192,87482X10,2015-03-18,0.07,87482X1020150318,156.44,0.000447,2015,1,201503
1193,87482X10,2015-06-18,0.04,87482X1020150618,150.53,0.000266,2015,2,201506


## Step 4: Map CUSIP to PERMNO

If we want to obtain the PERMNO corresponding to each stock, we can read "cusip.csv" where a mapping between CUSIP and PERMNO is provided. 

In [21]:
df4=pd.read_csv("classdata/cusip.csv")
df4["PERMNO"]=df4["PERMNO"].astype(str)
df4.head()

,PERMNO,NCUSIP
0,93433,87482X10
1,32548,FA0DT0WJ1
2,10421,AJB3NO4A2
3,10662,NQQXRB3M9
4,11757,Z8815JJD3


To do so, we just need to create a dictionary that maps NCUSIP to PERMNO. 

In [22]:
dictionary=dict(zip(df4.NCUSIP,df4.PERMNO))

In [23]:
df3["PERMNO"]=df3.CUSIP.map(dictionary)
df3[["PERMNO","CUSIP","Date","Signal"]].head()

,PERMNO,CUSIP,Date,Signal
0,31560,017VGPE63,201303,0.000061
1,31560,017VGPE63,201306,0.000216
2,31560,017VGPE63,201309,0.000062
3,31560,017VGPE63,201312,0.000133
4,31560,017VGPE63,201403,0.000126


In [24]:
df3.dropna(how='any',axis=0, inplace=True)
df3.reset_index(drop=True,inplace=True)
df3[["PERMNO","CUSIP","STATPERS","Date","Signal"]].head()

,PERMNO,CUSIP,STATPERS,Date,Signal
0,31560,017VGPE63,2013-03-18,201303,0.000061
1,31560,017VGPE63,2013-06-18,201306,0.000216
2,31560,017VGPE63,2013-09-18,201309,0.000062
3,31560,017VGPE63,2013-12-18,201312,0.000133
4,31560,017VGPE63,2014-03-18,201403,0.000126


## Step 5: Generate Summary Statistics

In [25]:
df3[df3.quarter==4].groupby("Date")["Signal"].describe(percentiles=[0.1,0.9])

,count,mean,std,min,10%,50%,90%,max
Date,,,,,,,,
201312,300.0,0.000496,0.000721,0.000029,0.000073,0.000273,0.000984,0.005090
201412,300.0,0.000508,0.000963,0.000028,0.000081,0.000257,0.000905,0.010920
201512,300.0,0.000502,0.000976,0.000030,0.000087,0.000264,0.000954,0.012403
201612,300.0,0.000486,0.000756,0.000029,0.000099,0.000280,0.000949,0.008837


Save the columns we want to a csv file.

In [26]:
df3[["PERMNO","STATPERS","Date","Signal"]].to_csv("Signal2.csv",index=False)